# Échantillonnage d'appels Aircall pour évaluation

Ce notebook extrait aléatoirement 5 appels par agent (Aircall) selon les critères suivants:

- **InCallDuration > 5 minutes**
- **IVR Branch = "Stellair"**
- **Filtre par mois** via le champ `Mois` (format `YYYY-MM`)

Il propose également l'export des échantillons vers Excel.


In [ ]:
# Configuration du chemin du projet pour permettre `import data_processing` et centraliser les chemins
import os, sys

def find_project_root(start_dir=None):
    start = os.path.abspath(start_dir or os.getcwd())
    cur = start
    while True:
        # condition: présence du dossier 'data_processing' à la racine du projet
        if os.path.isdir(os.path.join(cur, 'data_processing')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return None
        cur = parent

project_root = find_project_root()
if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

# Centralisation des chemins par défaut en fonction de project_root
DEFAULT_CACHE_DIR = os.path.join(project_root, 'data', 'Affid', 'Cache') if project_root else os.path.join('data','Affid','Cache')
DEFAULT_SQLITE_DB = os.path.join(DEFAULT_CACHE_DIR, 'cache.sqlite')
DEFAULT_PARQUET = os.path.join(DEFAULT_CACHE_DIR, 'aircall_processed.parquet')
DEFAULT_CSV = os.path.join(DEFAULT_CACHE_DIR, 'aircall_processed.csv')
DEFAULT_DF_SUPPORT = os.path.join(DEFAULT_CACHE_DIR, 'df_support.parquet')
DEFAULT_PROCESSED_PKL = os.path.join(DEFAULT_CACHE_DIR, 'processed_cache.pkl')

print('Project root:', project_root)
print('Default cache dir:', DEFAULT_CACHE_DIR)


Project root: c:\Users\ChristopheBRICHET\OneDrive - OLAQIN\Python\depot_new
Default cache dir: c:\Users\ChristopheBRICHET\OneDrive - OLAQIN\Python\depot_new\data\Affid\Cache


In [23]:
# Configuration des sources de données
# Option 1: laisser None pour auto-détection (parquet/csv puis SQLite)
# Option 2: fournir un chemin explicite vers un fichier parquet/csv contenant les colonnes requises
CUSTOM_PATH = None  # ex: 'C:/path/vers/aircall_processed.parquet' ou .csv

# Paramètres SQLite (si cache utilisé par l'app)
SQLITE_DB_PATH = DEFAULT_SQLITE_DB
SQLITE_TABLE = 'aircall_processed'  # à adapter si différent

# Fichiers par défaut
DEFAULT_FILES = [DEFAULT_PARQUET, DEFAULT_CSV]


In [37]:
# Chargement robuste des données (utiliser cette cellule)

import sqlite3

df_aircall = None

# 1) CUSTOM_PATH explicite
if CUSTOM_PATH is not None:
    if not os.path.exists(CUSTOM_PATH):
        raise FileNotFoundError(f"CUSTOM_PATH introuvable: {CUSTOM_PATH}")
    if CUSTOM_PATH.endswith('.parquet'):
        df_aircall = pd.read_parquet(CUSTOM_PATH)
    else:
        df_aircall = pd.read_csv(CUSTOM_PATH)
    print(f"Données chargées depuis CUSTOM_PATH: {CUSTOM_PATH}")

# 2) chemins standards (si pas de CUSTOM_PATH)
if df_aircall is None:
    for p in DEFAULT_FILES:
        if os.path.exists(p):
            if p.endswith('.parquet'):
                df_aircall = pd.read_parquet(p)
            else:
                df_aircall = pd.read_csv(p)
            print(f"Données chargées depuis: {p}")
            break

# 3) SQLite cache
if df_aircall is None and os.path.exists(SQLITE_DB_PATH):
    try:
        with sqlite3.connect(SQLITE_DB_PATH) as conn:
            df_aircall = pd.read_sql_query(f"SELECT * FROM {SQLITE_TABLE}", conn)
        print(f"Données chargées depuis SQLite: {SQLITE_DB_PATH} (table {SQLITE_TABLE})")
    except Exception as e:
        print("Lecture SQLite échouée:", e)

# 4) Fallback: df_support.parquet
if df_aircall is None and os.path.exists(DEFAULT_DF_SUPPORT):
    try:
        df_tmp = pd.read_parquet(DEFAULT_DF_SUPPORT)
        # Vérifier les colonnes requises
        required_cols = ['UserName', 'InCallDuration', 'IVR Branch', 'Mois']
        if all(c in df_tmp.columns for c in required_cols):
            df_aircall = df_tmp
            print(f"Données chargées depuis: {DEFAULT_DF_SUPPORT}")
        else:
            print(f"{DEFAULT_DF_SUPPORT} ne contient pas toutes les colonnes requises: {required_cols}")
    except Exception as e:
        print("Lecture df_support.parquet échouée:", e)

# 5) Fallback: processed_cache.pkl (dict picklé)
if df_aircall is None and os.path.exists(DEFAULT_PROCESSED_PKL):
    try:
        cache_obj = pd.read_pickle(DEFAULT_PROCESSED_PKL)
        # Chercher un DataFrame pertinent dans le pickle
        candidates = []
        if isinstance(cache_obj, dict):
            for k, v in cache_obj.items():
                if isinstance(v, pd.DataFrame) and all(col in v.columns for col in ['UserName','InCallDuration']):
                    candidates.append((k, v))
        if candidates:
            # Prendre le premier candidat qui a aussi 'IVR Branch' et 'Mois'
            chosen = None
            for k, v in candidates:
                if all(col in v.columns for col in ['IVR Branch','Mois']):
                    chosen = (k, v)
                    break
            if chosen is None:
                chosen = candidates[0]
                # Compléter 'Mois' si possible via 'StartTime'
                v = chosen[1]
                if 'Mois' not in v.columns and 'StartTime' in v.columns:
                    v = v.copy()
                    v['Mois'] = pd.to_datetime(v['StartTime']).dt.strftime('%Y-%m')
                df_aircall = v
            else:
                df_aircall = chosen[1]
            print(f"Données chargées depuis processed_cache.pkl (clé: {chosen[0] if 'chosen' in locals() and chosen else 'inconnue'})")
        else:
            print("Aucun DataFrame compatible trouvé dans processed_cache.pkl")
    except Exception as e:
        print("Lecture processed_cache.pkl échouée:", e)

if df_aircall is None:
    raise FileNotFoundError(f"Aucune donnée Aircall traitée trouvée.\n- Définissez CUSTOM_PATH\n- Ou placez {os.path.basename(DEFAULT_PARQUET)} / {os.path.basename(DEFAULT_CSV)} dans {DEFAULT_CACHE_DIR}\n- Ou assurez-vous que {SQLITE_DB_PATH} existe et contient la table {SQLITE_TABLE}\n- Ou fournissez {os.path.basename(DEFAULT_DF_SUPPORT)} / {os.path.basename(DEFAULT_PROCESSED_PKL)} exploitables.")

# Normaliser/garantir les colonnes nécessaires
if 'Mois' not in df_aircall.columns and 'StartTime' in df_aircall.columns:
    df_aircall['Mois'] = pd.to_datetime(df_aircall['StartTime']).dt.strftime('%Y-%m')

required_cols = ['UserName', 'InCallDuration', 'IVR Branch', 'Mois']
missing = [c for c in required_cols if c not in df_aircall.columns]
if missing:
    raise KeyError(f"Colonnes manquantes dans df_aircall: {missing}")

df_aircall = df_aircall[df_aircall['InCallDuration'].notna()]


df_aircall.head()


c:\Users\ChristopheBRICHET\OneDrive - OLAQIN\Python\depot_new\data\Affid\Cache\df_support.parquet ne contient pas toutes les colonnes requises: ['UserName', 'InCallDuration', 'IVR Branch', 'Mois']
Données chargées depuis processed_cache.pkl (clé: data)


,line,Semaine,Date,Jour,Heure,direction,LastState,ScenarioName,StartTime,HangupTime,time (TZ offset incl.),TotalDuration,InCallDuration,FromNumber,ToNumber,UserName,Tags,IVR Branch,Mois
9976,Armatis Technique,S2024-01,2024-12-30,Monday,10.0,inbound,yes,NaN,2024-12-30,1900-01-01 10:54:01,1900-01-01 10:49:37,593.000,264.0,33770640500,33189718190,Morgane Vandenbussche,STEFacturation,NaN,2024-12
9538,Technique,S2024-01,2024-12-31,Tuesday,16.0,inbound,yes,NaN,2024-12-31,1900-01-01 16:27:32,1900-01-01 16:12:50,1.038,882.0,33474315075,33189730123,Mourad HUMBLOT,nan,Stellair,2024-12
9539,Standard,S2024-01,2024-12-31,Tuesday,16.0,outbound,yes,NaN,2024-12-31,1900-01-01 16:23:10,1900-01-01 16:11:46,690.000,684.0,33187662300,33788944440,Olivier Sainte-Rose,STECB,NaN,2024-12
9540,Technique,S2024-01,2024-12-31,Tuesday,16.0,outbound,yes,NaN,2024-12-31,1900-01-01 16:14:41,1900-01-01 16:09:32,315.000,309.0,33189730123,33676622951,Mourad HUMBLOT,STERDV,NaN,2024-12
9541,Standard,S2024-01,2024-12-31,Tuesday,15.0,outbound,yes,NaN,2024-12-31,1900-01-01 16:10:52,1900-01-01 15:59:28,703.000,684.0,33187662300,33788944440,Olivier Sainte-Rose,STECB,NaN,2024-12


In [38]:
# Imports et configuration
import os
import pandas as pd
from datetime import datetime

# Chargement des fonctions existantes du projet
try:
    from data_processing.aircall_processing import process_aircall_data, def_df_support
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("Impossible d'importer 'data_processing'. Exécutez d'abord la cellule précédente pour configurer sys.path ou lancez le notebook depuis la racine du projet.")

# Option d'affichage Pandas
pd.set_option('display.max_columns', None)


In [ ]:
def choisir_enregistrements_aleatoires(df, username, mois, branche='Stellair', duree_min=300, n=5):
    """
    Filtre le DataFrame selon les critères spécifiés et retourne n enregistrements tirés aléatoirement.
    
    Paramètres :
        df (pd.DataFrame) : le DataFrame d'origine
        username (str) : le nom de l'agent
        mois (str) : le mois au format 'YYYY-MM'
        branche (str) : le nom de la branche IVR (par défaut 'Stellair')
        duree_min (int) : durée minimale de communication en secondes (par défaut 300s)
        n (int) : nombre d'enregistrements à retourner (par défaut 5)

    Retour :
        pd.DataFrame : un DataFrame de n lignes choisies aléatoirement
    """
    filtres = (
        (df['Mois'] == mois) &
        (df['InCallDuration'] > duree_min) &
        (df['IVR Branch'] == branche) &
        (df['UserName'] == username)
    )
    
    df_filtre = df[filtres]

    # Si moins de n résultats, retourne tout
    return df_filtre.sample(n=min(n, len(df_filtre)), random_state=42)

In [ ]:
# Échantillonnage: 5 appels aléatoires par agent

# On groupe par 'UserName' et on échantillonne 5 lignes par groupe, si possible

def sample_n_per_group(df, group_col, n=5, random_state=None):
    def _sampler(g):
        if len(g) <= n:
            return g.sample(len(g), random_state=random_state)
        return g.sample(n, random_state=random_state)
    return df.groupby(group_col, group_keys=False).apply(_sampler)

sampled = sample_n_per_group(sample_base, 'UserName', n=5, random_state=42)
print(sampled.groupby('UserName').size())

sampled.head(10)


In [ ]:
# Export Excel (optionnel)

export_dir = 'exports'
os.makedirs(export_dir, exist_ok=True)

export_path = os.path.join(export_dir, f"echantillon_aircall_{mois_choisi}.xlsx")
with pd.ExcelWriter(export_path, engine='xlsxwriter') as writer:
    sampled.to_excel(writer, index=False, sheet_name='Echantillon')
    sample_base.to_excel(writer, index=False, sheet_name='BaseFiltree')

print(f"Fichier exporté: {export_path}")
